# Exercise 5 · Write the evaluation report

**Part 06 · Benchmarking and evaluation** · 10 minutes

Decide whether your model should be **deployed**, run in **shadow** mode, or **not deployed**, and back
it with numbers you computed yourself.

Data: `outputs/predictions_demo5.jsonl` from Demo 5. If you have not run Demo 5, the notebook uses the
saved benchmark predictions (Jev, Kev-4B, Laya on AG News) instead, and you evaluate Kev-4B.

In [ ]:
# Setup: find the workshop folder and import the helpers (works locally and in Colab)
import os, sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/decision-models-workshop")]
             if (p / "dmw").is_dir()), None)
assert ROOT, "Open this notebook from the workshop folder. In Colab, run 00_setup.ipynb first."
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from dmw import config
config.load_env()

In [ ]:
import json
import pandas as pd
from dmw import metrics, checks, bench

DEMO5 = config.OUTPUTS / "predictions_demo5.jsonl"
if DEMO5.exists():
    SOURCE, P = "demo5", pd.read_json(DEMO5, lines=True)
    MODEL, OTHER = "fine-tuned", ("jev" if "jev" in set(pd.read_json(DEMO5, lines=True).model) else "base")
else:
    SOURCE = "benchmark_replay"
    f = bench.frame("ag_news")
    P = pd.concat([pd.DataFrame({"model": m, "id": f.id, "group": "ag_news", "expected": f.gold,
                                 "choice": f[m], "p_top": f[f"{m}_p"], "latency_ms": f[f"{m}_ms"]})
                   for m in ("jev", "kev-4b", "laya")])
    P["review"] = P.p_top.fillna(0) < 0.8
    MODEL, OTHER = "kev-4b", "jev"
print("SOURCE =", SOURCE, "| evaluating", MODEL, "against", OTHER)
P.groupby("model").size()

## 1 · Metrics for your model

In [ ]:
d = P[P.model == MODEL]
accuracy = None              # TODO: metrics.accuracy(gold, predicted)
macro_f1 = None              # TODO
accuracy_not_flagged = None  # TODO: accuracy on the rows where review is False
print(accuracy, macro_f1, accuracy_not_flagged)

## 2 · Coverage vs automatic errors

In [ ]:
curve = metrics.coverage_curve(d.p_top.fillna(0).tolist(), (d.choice == d.expected).tolist())
curve

## 3 · Disagreements with the other decider

In [ ]:
w = P.pivot(index="id", columns="model", values="choice").join(P[P.model == MODEL].set_index("id")[["expected"]])
w[w[MODEL] != w[OTHER]]

## 4 · Your report

In [ ]:
REPORT = {
    "decision": "",            # TODO: "deploy", "shadow" or "do_not_deploy"
    "threshold": None,         # TODO: from the coverage curve
    "accuracy": accuracy, "macro_f1": macro_f1, "accuracy_not_flagged": accuracy_not_flagged,
    "disagreements": [
        # TODO: three of them: {"id": ..., "who_is_right": ..., "likely_cause": "..."}
    ],
    "missing_evidence": "",    # TODO: what would you need to see before trusting this in production?
}
checks.check_report(REPORT, checks.recompute(P, MODEL))

## 5 · Optional: add the evaluation to your model card

In [ ]:
PUSH_TO_CARD = False
if PUSH_TO_CARD and SOURCE == "demo5":
    from huggingface_hub import HfApi
    repo = f"{config.key('HF_USERNAME')}/qwen3-0.6b-decision-router"
    api = HfApi(token=config.key("HF_TOKEN"))
    readme = open(api.hf_hub_download(repo, "README.md")).read()
    section = "\n## Evaluation report (workshop)\n\n" + json.dumps(REPORT, indent=1, default=str)[:3000] + "\n"
    api.upload_file(path_or_fileobj=(readme + section).encode(), path_in_repo="README.md", repo_id=repo,
                    commit_message="Add evaluation report")
    print("Updated", repo)